# PDF RAG Chatbot

Hybrid search (BM25 + TF-IDF) + LLM reranking + query rewriting, using Groq's Llama 3.3 70B.

**Setup before running:**
1. Put your PDF files inside the `pdfs/` folder
2. Copy `.env.example` to `.env` and add your Groq API key
3. Run `pip install -r requirements.txt`

In [ ]:
import os
import numpy as np
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # reads variables from a local .env file
print("Libraries loaded")

In [ ]:
# ---------- CONFIG ----------
CHUNK_SIZE = 500
TOP_K = 8

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError(
        "GROQ_API_KEY not found. Create a .env file (see .env.example) "
        "and add your key there, or export it as an environment variable."
    )

client = Groq(api_key=GROQ_API_KEY)

def llm_call(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content

print("Config ready")

In [ ]:
def chunk_text(text, size=CHUNK_SIZE):
    """Split text into word-based chunks of fixed size."""
    words = text.split()
    return [" ".join(words[i:i + size]) for i in range(0, len(words), size)]

In [ ]:
def read_pdfs(folder):
    """Read every PDF in `folder`, chunk it, and attach source/page metadata."""
    docs = []
    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            path = os.path.join(folder, file)
            reader = PdfReader(path)
            for i, page in enumerate(reader.pages):
                text = page.extract_text()
                if text:
                    for chunk in chunk_text(text):
                        docs.append({
                            "text": chunk,
                            "metadata": {"source": file, "page": i}
                        })
    return docs

In [ ]:
tfidf = None  # set by build_tfidf()

def build_tfidf(docs):
    global tfidf
    corpus = [d["text"] for d in docs]
    tfidf = TfidfVectorizer(max_features=5000)
    tfidf.fit(corpus)
    print("TF-IDF ready")

def get_embedding(text):
    return tfidf.transform([text]).toarray()[0]

In [ ]:
def build_index(docs):
    corpus = [d["text"] for d in docs]

    tokenized = [doc.lower().split() for doc in corpus]
    bm25 = BM25Okapi(tokenized)

    embeddings = tfidf.transform(corpus).toarray()

    return bm25, embeddings

In [ ]:
def rewrite_query(query):
    """Turn a natural-language question into a short keyword query for better retrieval."""
    prompt = f"""Convert this query into 3-5 important keywords only for document search.
Return ONLY the keywords separated by spaces.
No sentences, no explanation, no punctuation.

Query: {query}

Keywords:"""
    result = llm_call(prompt).strip().split("\n")[0].strip()
    return result

In [ ]:
def hybrid_search(query, docs, bm25, embeddings):
    """Combine BM25 (keyword) and TF-IDF cosine similarity scores."""
    tokenized_query = query.lower().split()

    bm25_scores = bm25.get_scores(tokenized_query)
    if bm25_scores.max() > 0:
        bm25_scores = bm25_scores / bm25_scores.max()

    query_emb = get_embedding(query).reshape(1, -1)
    emb_scores = cosine_similarity(query_emb, embeddings)[0]

    scores = 0.7 * bm25_scores + 0.3 * emb_scores

    top_idx = np.argsort(scores)[::-1][:TOP_K]
    return [docs[i] for i in top_idx]

In [ ]:
def rerank(query, docs):
    """Ask the LLM to reorder retrieved chunks by relevance."""
    chunks_text = ""
    for i, d in enumerate(docs):
        chunks_text += f"Chunk {i}:\n{d['text'][:300]}\n\n"

    prompt = f"""You are a reranking assistant.
Given a query and multiple chunks, return ONLY a Python list of chunk indices
ranked from most to least relevant.
Example output: [2, 0, 4, 1, 3]
Return ONLY the list, nothing else.

Query: {query}

Chunks:
{chunks_text}"""

    try:
        result = llm_call(prompt).strip()
        ranked_indices = [int(x) for x in result.strip("[]").split(",") if x.strip().isdigit()]
        return [docs[i] for i in ranked_indices if i < len(docs)] or docs
    except Exception:
        return docs

In [ ]:
def generate_answer(query, docs):
    context = ""
    for d in docs[:3]:
        context += f"[Source: {d['metadata']['source']}, Page: {d['metadata']['page']}]\n"
        context += d["text"] + "\n\n"

    prompt = f"""You are an expert teacher. Answer the question using the context below.

Rules:
1. Take the DEFINITION strictly from the context only
2. Create 2-3 easy examples YOURSELF to explain better
3. Give answer in this exact format:

**Definition:** (from context)

**Explanation:** (in simple words)

**Examples:**
- Example 1: ...
- Example 2: ...
- Example 3: ...

If definition is not in context at all, say "I don't know, this is not in the document."

Context:
{context}

Question: {query}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=1024
    )
    return response.choices[0].message.content

In [ ]:
def chat(query, docs, bm25, embeddings):
    new_query = rewrite_query(query)
    retrieved = hybrid_search(new_query, docs, bm25, embeddings)
    reranked = rerank(new_query, retrieved)
    return generate_answer(query, reranked)

## Build the index

Run this once (or whenever you add new PDFs to `pdfs/`).

In [ ]:
PDF_FOLDER = "pdfs"

print("Indexing PDFs...")
docs = read_pdfs(PDF_FOLDER)
print(f"Total chunks: {len(docs)}")

build_tfidf(docs)
bm25, embeddings = build_index(docs)
print("Ready! Ask questions below.")

## Chat loop

Type a question, type `exit` to quit.

In [ ]:
while True:
    q = input(">> ")
    if q.lower() == "exit":
        print("Bye!")
        break
    ans = chat(q, docs, bm25, embeddings)
    print(f"\nAnswer: {ans}\n")
    print("=" * 50)